# Module 11: Production Feature Selection Pipelines

## Learning Objectives

By completing this notebook, you will:
1. Build a custom `StabilitySelector` transformer that implements the sklearn `fit`/`transform` interface
2. Compose a complete `Pipeline` with preprocessing, stability selection, and a gradient boosting model
3. Cross-validate the entire pipeline correctly — with selection re-run inside each fold
4. Serialise the fitted pipeline to disk and reload it, verifying that the selection state is preserved
5. Demonstrate the difference between correct pipeline CV and the leaky anti-pattern

## Prerequisites
- sklearn Pipeline basics (Guide 01)
- Stability selection concept (Module 10)
- NumPy, pandas, matplotlib

## Estimated Time: 15 minutes

---

## 1. Setup

We use the California housing dataset as a binary classification problem: predict whether median house value is above the dataset median. This gives us a realistic tabular dataset with 8 features — sufficient to illustrate pipeline mechanics without being overwhelming.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import pathlib
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.utils import resample
from sklearn.utils.validation import check_is_fitted
from sklearn.metrics import roc_auc_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

# Load and binarise California housing
housing = fetch_california_housing(as_frame=True)
X = housing.frame.drop(columns=['MedHouseVal'])
y = (housing.frame['MedHouseVal'] > housing.frame['MedHouseVal'].median()).astype(int)

print(f"Dataset: {X.shape[0]:,} rows, {X.shape[1]} features")
print(f"Features: {list(X.columns)}")
print(f"Target distribution: {y.mean():.2%} positive")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")

## 2. The StabilitySelector Transformer

Stability selection answers a critical production question: **"Would this feature be selected if we had slightly different training data?"**

It works by:
1. Drawing many bootstrap subsamples of the training data (typically 50% each)
2. Running the base selector on each subsample
3. Keeping only features selected in at least `threshold` fraction of runs

Features with high selection frequency are **robust to sampling variation** — they represent genuine signal. Features with low frequency are noise artifacts that happen to be selected on one particular dataset.

We wrap this as a sklearn `TransformerMixin` so it drops directly into a `Pipeline`.

In [ ]:
class StabilitySelector(BaseEstimator, TransformerMixin):
    """
    Stability selection: retain features selected in >= threshold fraction
    of bootstrap subsamples.

    Parameters
    ----------
    base_selector : sklearn selector
        Must implement fit(X, y) and get_support().
        Will be cloned before each bootstrap run.
    n_bootstrap : int
        Number of bootstrap iterations (more = more stable estimates).
    threshold : float in (0.5, 1.0]
        Minimum selection frequency for a feature to be kept.
    subsample : float in (0, 1)
        Fraction of training data per bootstrap draw.
    random_state : int | None
        Seed for reproducibility.
    """

    def __init__(
        self,
        base_selector,
        n_bootstrap: int = 50,
        threshold: float = 0.6,
        subsample: float = 0.5,
        random_state: int = 42,
    ):
        self.base_selector = base_selector
        self.n_bootstrap   = n_bootstrap
        self.threshold     = threshold
        self.subsample     = subsample
        self.random_state  = random_state

    def fit(self, X, y):
        """Run bootstrap stability selection on training data."""
        rng = np.random.default_rng(self.random_state)

        # Convert to arrays for resample compatibility
        X_arr = X.values if hasattr(X, 'values') else np.asarray(X)
        y_arr = np.asarray(y)
        n_features = X_arr.shape[1]
        n_subsample = max(10, int(len(y_arr) * self.subsample))

        # Count how many times each feature is selected across bootstrap runs
        selection_counts = np.zeros(n_features, dtype=int)

        for i in range(self.n_bootstrap):
            seed = int(rng.integers(0, 2**31))
            X_boot, y_boot = resample(
                X_arr, y_arr, n_samples=n_subsample, random_state=seed
            )
            # Clone the selector to get a fresh, unfitted copy each iteration
            sel = clone(self.base_selector)
            sel.fit(X_boot, y_boot)
            selection_counts += sel.get_support().astype(int)

        # Compute selection frequency and apply threshold
        self.selection_scores_ = selection_counts / self.n_bootstrap
        self.support_ = self.selection_scores_ >= self.threshold
        self.n_features_in_ = n_features

        # Store feature names for get_feature_names_out
        if hasattr(X, 'columns'):
            self.feature_names_in_ = np.array(X.columns)
        else:
            self.feature_names_in_ = np.array([f'x{i}' for i in range(n_features)])

        return self  # always return self

    def transform(self, X):
        """Apply the fitted selection mask."""
        check_is_fitted(self, 'support_')  # raises NotFittedError if not fitted
        if hasattr(X, 'iloc'):
            return X.iloc[:, self.support_]
        return np.asarray(X)[:, self.support_]

    def get_feature_names_out(self, input_features=None):
        """Return names of the selected features."""
        check_is_fitted(self, 'support_')
        return self.feature_names_in_[self.support_]

    def get_selection_scores(self) -> pd.Series:
        """Per-feature selection frequency, sorted descending."""
        check_is_fitted(self, 'selection_scores_')
        return pd.Series(
            self.selection_scores_,
            index=self.feature_names_in_,
        ).sort_values(ascending=False)

print("StabilitySelector defined successfully.")

## 3. Build the Production Pipeline

We compose three steps into a single estimator:

1. **`StandardScaler`**: Normalises features to zero mean, unit variance. Required before any distance-based or gradient-based method.
2. **`StabilitySelector`**: Bootstrap stability selection wrapping a `SelectFromModel(RandomForestClassifier)` base. Only features selected in >= 60% of 50 bootstrap runs pass through.
3. **`GradientBoostingClassifier`**: The final model trained on only the selected features.

The `Pipeline` object handles all orchestration. During `fit`, each step calls `fit_transform` in sequence. During `predict`, each step except the last calls `transform`.

In [ ]:
# Base selector: keep features above mean importance according to a small RF
base_selector = SelectFromModel(
    estimator=RandomForestClassifier(n_estimators=50, random_state=42),
    threshold='mean',
)

# Wrap in stability selection
stability_selector = StabilitySelector(
    base_selector=base_selector,
    n_bootstrap=50,      # more = more stable; 50 is fast for demo
    threshold=0.6,       # feature must be selected in >= 60% of runs
    subsample=0.5,       # use 50% of training data per bootstrap
    random_state=42,
)

# Compose the full pipeline
pipeline = Pipeline([
    ('scaler',   StandardScaler()),
    ('selector', stability_selector),
    ('model',    GradientBoostingClassifier(
        n_estimators=100, max_depth=3, random_state=42
    )),
])

print("Pipeline steps:")
for step_name, step_obj in pipeline.steps:
    print(f"  {step_name}: {type(step_obj).__name__}")

## 4. Correct Cross-Validation: Selection Inside the Pipeline

The single most important thing about pipeline CV: **each fold runs its own independent feature selection on that fold's training data**. The validation fold never influences which features are selected.

Compare two approaches:
- **Correct**: `cross_validate(pipeline, X, y, cv=cv)` — selection inside each fold
- **Leaky**: Select features first, then CV the model — selection saw all folds

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Running correct pipeline cross-validation...")
print("(Each fold re-runs stability selection on that fold's training data.)")
print()

# Correct: pipeline handles all orchestration per fold
results_correct = cross_validate(
    pipeline, X_train, y_train,
    cv=cv,
    scoring='roc_auc',
    return_train_score=True,
    n_jobs=-1,
)

print(f"Pipeline CV (correct):")
print(f"  Train AUC: {results_correct['train_score'].mean():.4f} "
      f"± {results_correct['train_score'].std():.4f}")
print(f"  Val AUC:   {results_correct['test_score'].mean():.4f} "
      f"± {results_correct['test_score'].std():.4f}")

In [ ]:
# Leaky anti-pattern: select on full X_train THEN cross-validate
print("Running leaky anti-pattern for comparison...")
print("(Selector fit on ALL of X_train before any fold split.)")
print()

# Fit selector on all training data — this leaks information into validation folds
leaky_selector = StabilitySelector(
    base_selector=SelectFromModel(
        RandomForestClassifier(n_estimators=50, random_state=42), threshold='mean'
    ),
    n_bootstrap=50, threshold=0.6, random_state=42,
)
leaky_selector.fit(X_train, y_train)
X_train_leaky = leaky_selector.transform(X_train)

# Now CV the model on pre-selected features
from sklearn.model_selection import cross_val_score
leaky_model = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
leaky_scores = cross_val_score(
    leaky_model, X_train_leaky, y_train, cv=cv, scoring='roc_auc'
)

print(f"Leaky CV (anti-pattern):")
print(f"  Val AUC: {leaky_scores.mean():.4f} ± {leaky_scores.std():.4f}")
print()
print(f"Correct pipeline Val AUC: {results_correct['test_score'].mean():.4f}")
print(f"Leaky anti-pattern AUC:   {leaky_scores.mean():.4f}")
print(f"Optimism bias from leakage: "
      f"+{leaky_scores.mean() - results_correct['test_score'].mean():.4f}")

The leaky approach reports a higher validation AUC because the selector used validation fold data during fit — it implicitly optimised the feature set for those validation examples. This bias will not materialise in true out-of-sample performance, leading to deployment disappointment.

## 5. Fit on Full Training Data and Inspect Selection State

After confirming via CV that the pipeline is well-specified, fit it on the full training set to produce the deployment artefact.

In [ ]:
print("Fitting pipeline on full training data...")
pipeline.fit(X_train, y_train)

# Inspect the fitted selector
fitted_selector = pipeline.named_steps['selector']
scores = fitted_selector.get_selection_scores()

print(f"\nStability selection results:")
print(f"  Features considered: {len(scores)}")
print(f"  Features selected:   {fitted_selector.support_.sum()}")
print(f"  Selection threshold: {fitted_selector.threshold}")
print()
print("Per-feature selection frequencies:")
for feat, freq in scores.items():
    marker = '  SELECTED' if freq >= fitted_selector.threshold else '  rejected'
    print(f"  {feat:20s}: {freq:.2f} {marker}")

# Evaluate on held-out test set
y_prob = pipeline.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, y_prob)
print(f"\nHeld-out test AUC: {test_auc:.4f}")

In [ ]:
# Visualise selection frequencies
fig, ax = plt.subplots(figsize=(10, 4))

colors = ['#2ca02c' if freq >= fitted_selector.threshold else '#d62728'
          for freq in scores.values]
ax.barh(scores.index, scores.values, color=colors, edgecolor='white')
ax.axvline(x=fitted_selector.threshold, color='black', linestyle='--',
           linewidth=1.5, label=f'Threshold ({fitted_selector.threshold})')
ax.set_xlabel('Selection Frequency (fraction of bootstrap runs)', fontsize=12)
ax.set_title('Stability Selection: Feature Selection Frequencies', fontsize=13)
ax.legend()
ax.set_xlim(0, 1.05)
plt.tight_layout()
plt.show()

print("Green bars: selected features (frequency >= threshold)")
print("Red bars:   rejected features (frequency < threshold)")

## 6. Serialise and Reload the Pipeline

A pipeline is only useful in production if it can be serialised to disk and reloaded in a fresh process. `joblib` is preferred over `pickle` for sklearn objects because it handles large numpy arrays efficiently via memory mapping.

The critical check: after reloading, predictions must be **byte-identical** to pre-serialisation predictions. If they differ, something in the pipeline's state was not properly serialised.

In [ ]:
# Create output directory
artefact_dir = pathlib.Path('artefacts')
artefact_dir.mkdir(exist_ok=True)
model_path = artefact_dir / 'feature_pipeline_v1.pkl'

# Serialise
joblib.dump(pipeline, model_path)
file_size_kb = model_path.stat().st_size / 1024
print(f"Pipeline serialised to: {model_path}")
print(f"File size: {file_size_kb:.1f} KB")

# Reload in the same process (simulates a fresh deployment)
loaded_pipeline = joblib.load(model_path)

# Verify selection state is preserved
loaded_selector = loaded_pipeline.named_steps['selector']
original_selected = set(fitted_selector.get_feature_names_out())
loaded_selected   = set(loaded_selector.get_feature_names_out())

print(f"\nSelection state preserved after reload:")
print(f"  Original selected features: {sorted(original_selected)}")
print(f"  Loaded selected features:   {sorted(loaded_selected)}")
print(f"  Sets identical: {original_selected == loaded_selected}")

# Verify predictions are identical
y_prob_original = pipeline.predict_proba(X_test)[:, 1]
y_prob_loaded   = loaded_pipeline.predict_proba(X_test)[:, 1]

predictions_match = np.allclose(y_prob_original, y_prob_loaded, rtol=1e-10)
print(f"\nPredictions byte-identical: {predictions_match}")
print(f"Max absolute difference: {np.abs(y_prob_original - y_prob_loaded).max():.2e}")

if predictions_match:
    print("\nSerialisation verified — pipeline is production-ready.")
else:
    print("\nWARNING: Predictions differ after reload. Investigate before deploying.")

## 7. Exercise: Extend the Pipeline

**Task:** Modify the `StabilitySelector` to also support a **minimum feature count** constraint: never select fewer than `min_features` features, even if fewer pass the `threshold`. If too few features pass the threshold, relax the threshold iteratively by 0.05 until at least `min_features` are selected.

This is a common production requirement: some models require a minimum number of features to avoid degenerate behaviour (e.g., a model that always predicts the same class with only one feature).

**Hints:**
- After computing `self.selection_scores_`, check how many pass the threshold
- If fewer than `min_features`, reduce the effective threshold and recheck
- Store the final effective threshold as `self.effective_threshold_`

In [ ]:
class StabilitySelectorWithMinFeatures(BaseEstimator, TransformerMixin):
    """
    StabilitySelector with a minimum feature count guarantee.

    If fewer than min_features pass the threshold, the threshold is
    relaxed by 0.05 until at least min_features are selected.
    """

    def __init__(
        self,
        base_selector,
        n_bootstrap: int = 50,
        threshold: float = 0.6,
        min_features: int = 3,
        subsample: float = 0.5,
        random_state: int = 42,
    ):
        self.base_selector = base_selector
        self.n_bootstrap   = n_bootstrap
        self.threshold     = threshold
        self.min_features  = min_features
        self.subsample     = subsample
        self.random_state  = random_state

    def fit(self, X, y):
        # TODO: Implement fit using the same bootstrap loop as StabilitySelector
        # After computing self.selection_scores_, apply the min_features constraint:
        #   effective_threshold = self.threshold
        #   while (scores >= effective_threshold).sum() < self.min_features:
        #       effective_threshold -= 0.05
        #       if effective_threshold <= 0:
        #           break
        # Store effective_threshold as self.effective_threshold_
        raise NotImplementedError("Implement the fit method")

    def transform(self, X):
        check_is_fitted(self, 'support_')
        if hasattr(X, 'iloc'):
            return X.iloc[:, self.support_]
        return np.asarray(X)[:, self.support_]

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, 'support_')
        return self.feature_names_in_[self.support_]


# Self-check tests — run after implementing
def test_min_features_selector():
    from sklearn.linear_model import LogisticRegression
    from sklearn.feature_selection import SelectFromModel

    # Very high threshold — should force relaxation
    base = SelectFromModel(
        RandomForestClassifier(n_estimators=30, random_state=42), threshold='mean'
    )
    sel = StabilitySelectorWithMinFeatures(
        base_selector=base,
        n_bootstrap=30,
        threshold=0.95,    # very strict — may select fewer than min_features
        min_features=3,
        random_state=42,
    )
    sel.fit(X_train, y_train)

    n_selected = sel.support_.sum()
    assert n_selected >= 3, (
        f"Expected >= 3 features but got {n_selected}. "
        f"Check min_features constraint logic."
    )
    assert hasattr(sel, 'effective_threshold_'), (
        "Missing effective_threshold_ attribute. Store the final threshold used."
    )
    assert sel.effective_threshold_ <= sel.threshold, (
        "effective_threshold_ should be <= threshold (it can only be relaxed, not tightened)"
    )

    # Test that transform works
    X_transformed = sel.transform(X_train)
    assert X_transformed.shape[1] == n_selected, "transform column count mismatch"

    print(f"Test passed! {n_selected} features selected, "
          f"effective_threshold={sel.effective_threshold_:.2f} "
          f"(original={sel.threshold})")

# Uncomment to run after implementing:
# test_min_features_selector()

## 8. Summary

### Key Takeaways

1. **Every sklearn-compatible transformer** must implement `fit(X, y)`, `transform(X)`, and `get_feature_names_out()`. `BaseEstimator` and `TransformerMixin` provide `get_params`, `set_params`, and `fit_transform` for free.

2. **Stability selection** is the production-grade approach: bootstrap sampling exposes which features are noise artifacts (low frequency) vs. genuine signal (high frequency).

3. **Selection inside Pipeline** is non-negotiable. Cross-validating a model on pre-selected features leaks information across fold boundaries and produces optimistically biased scores.

4. **`joblib.dump`/`load`** serialises the entire pipeline state — scaler parameters, selection mask, and model weights — into a single `.pkl` file. Always verify with `np.allclose` after reload.

### What's Next

Notebook 02 covers **drift monitoring**: detecting when the feature distributions in production diverge from training, and triggering automated re-selection.

### Additional Resources
- sklearn Pipeline docs: https://scikit-learn.org/stable/modules/compose.html#pipeline
- Meinshausen & Bühlmann (2010): Stability Selection — *JRSS-B* 72(4)